# Create Ranked SoloQueue dataset

This notebook has not been used, becasue with the free tier of the riot api it is impossible to get all these games. I have just used the cell to obtain all the puuids and i got 3.620.690 puuids.

In [1]:
import requests
import os
import time
import json
from dotenv import load_dotenv

# cargamos los datos del .env
load_dotenv()

# obtenemos la api_key y la base_url
api_key = os.getenv("API_KEY")

## Obtain all the puuids from Solo/DuoQueue

### Obtain players from every tier

We have to go through every Elo and obtain all the players until the answer is []
https://euw1.api.riotgames.com/lol/league-exp/v4/entries/RANKED_SOLO_5x5/IRON/IV?page=1&api_key={api_key}

### Obtain high elo players

For high elo players (master, grandmaster, challenger) we use another call that gives us all the puuids from that elo
https://euw1.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}
https://euw1.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}
https://euw1.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}

In [ ]:
def get_from_api(url):
    while True:
        resp = requests.get(url)
        if resp.status_code == 429:
            wait_time = int(resp.headers['Retry-After'])
            display(f'Limit excedeed, waiting: {wait_time}')
            time.sleep(wait_time + 1)
            continue

        if resp.status_code != 200:
            display(f"Error {resp.status_code}")

        resp.raise_for_status()
        return resp.json()

def save_player_jsonl(players, default_tier, default_rank, saved_puuids, file):
    for player in players:
        puuid = player.get('puuid')
        if puuid and puuid not in saved_puuids:
            data = {
                'tier': player.get('tier') or default_tier,
                'rank': player.get('rank') or default_rank,
                'puuid': player.get('puuid')
            }

            file.write(json.dumps(data)+ "\n")
            saved_puuids.add(puuid)

    # para que si o si se guarde
    file.flush()

# crear arrays de los elos y de los ranks
elos = ["IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM", "EMERALD", "DIAMOND"]
ranks = ['IV', 'III', 'II', 'I']

grand_elos = ["master", "grandmaster", "challenger"]

page = 1

# set(sin duplicados) para guardar los puuids
saved_puuids = set()

base_url = "https://euw1.api.riotgames.com/lol"

with open("puuids_per_elo.jsonl", "a", encoding="utf-8") as file_puuids:
    for elo in elos:
        for rank in ranks:
            display(f"{elo} {rank}")
            page = 1

            while True:
                url = f"{base_url}/league-exp/v4/entries/RANKED_SOLO_5x5/{elo}/{rank}?page={page}&api_key={api_key}"
                data = get_from_api(url)

                # si no hay datos es pagina vacia y hay q cambiar de rango o elo
                if not data:
                    break

                save_player_jsonl(data, default_tier=elo, default_rank=rank, saved_puuids=saved_puuids, file=file_puuids)
                page += 1

    for grand_elo in grand_elos:
        display(f"Procesando Apex Tier: {grand_elo}")
        url = f"{base_url}/league/v4/{grand_elo.lower()}leagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}"
        data = get_from_api(url)
        
        tier = data.get('tier', grand_elo.upper())
        entries = data.get('entries', [])
        
        save_player_jsonl(entries, default_tier=tier, default_rank='I', saved_puuids=saved_puuids, file=file_puuids)

## Obtain match ids from specific patch (16.18)

This patch came out 10/09/2026 and ended in 22/09/2026, aprox.
What we do here is go through every puuid and search all the games between these 2 dates. In the other hand we use queue=420 to filter solo/duoQ games.
We save all the match_id in a txt without duplicates.

In [ ]:
base_url = f"https://europe.api.riotgames.com/lol/match/v5/matches/by-puuid/"

startTime = 1789000200 # epoch time (1789000200)
endTime = 1790123400
start = 0
count = 100

with open("puuids_per_elo.jsonl", "r", encoding="utf-8") as file_puuids:
    with open("match_to_process.txt", "a+", encoding="utf-8") as file_match_ids:
        # leemos el txt
        file_match_ids.seek(0) # nos movemos al principio
        matches = set(file_match_ids.read().splitlines())
        print(f"Partidas ya cargadas previamente: {len(matches)}")

        for line in file_puuids:
            if not line.strip():
                continue

            line_object = json.loads(line.strip())
            puuid = line_object.get('puuid')
            if not puuid:
                continue

            start = 0

            # para mirar todas las paginas hasta q nos devuelva []
            while True:
                resp = requests.get(f"{base_url}{puuid}/ids?startTime={startTime}&endTime={endTime}&queue=420&start={start}&count={count}&api_key={api_key}")

                if resp.status_code == 429:
                    wait_time = int(resp.headers['Retry-After'])
                    display(f'Limit excedeed, waiting: {wait_time}')
                    time.sleep(wait_time + 1)
                    continue

                match_ids = resp.json()

                # no hay mas partidas
                if not isinstance(match_ids, list) or len(match_ids) == 0:
                    break

                for match_id in match_ids:
                    if match_id and match_id not in matches:
                        file_match_ids.write(match_id + "\n")
                        matches.add(match_id)
                file_match_ids.flush()        

                start += count


## Obtain match data for games in patch (16.18)

We should check in 'info/gameversion' to get the patch version, so we can obtain all the patch dataset and save it.

In [ ]:
base_url = "https://europe.api.riotgames.com/lol/match/v5/matches/"

with open("processed_matches.txt", "r", encoding="utf-8") as file_processed_matches:
    matches_processats = set(file_processed_matches.read().splitlines())

print(f"Partidas ya descargadas: {len(matches_processats)}")

with open("match_to_process.txt", "r", encoding="utf-8") as file_match_ids:
    matches = set(file_match_ids.read().splitlines())
    
print(f"Total de partides úniques trobades: {len(matches)}")

# esto nos arregla tambien los sets
matches_left_to_process = matches - matches_processats
print(f"Partidas que faltan por descargar: {len(matches_left_to_process)}")

with open("matches_raw_dataset.jsonl", "a", encoding="utf-8") as file_match_raw_ds, \
     open("processed_matches.txt", "a", encoding="utf-8") as file_processed_write:
    for match_id in matches_left_to_process:
        if match_id == '':
            continue

        url = f"{base_url}{match_id}?api_key={api_key}"

        while True:
            resp = requests.get(url)

            if resp.status_code == 429:
                wait_time = int(resp.headers['Retry-After'])
                display(f'Limit excedeed, waiting: {wait_time}')
                time.sleep(wait_time + 1)
                continue

            break

        resp_json = resp.json()
        game_version = resp_json["info"]["gameVersion"]

        if resp.status_code == 200 and game_version.startswith("16.18"):
            file_match_raw_ds.write(json.dumps(resp_json)+ "\n")
            file_match_raw_ds.flush()

            file_processed_write.write(match_id + "\n")
            file_processed_write.flush()
        else:
            print(f"Error {resp.status_code} obteniendo la partida {match_id} con versión {game_version}")

Partidas ya descargadas: 0
Total de partides úniques trobades: 57
Partidas que faltan por descargar: 57
